# Library And Packages loading

In [ ]:
# Install the HuggingFace transformers library
!pip install -q transformers datasets torch
!pip install emoji

In [ ]:
# Packages import
import os
import pandas as pd
import numpy as np
import urllib.request
import re
from urllib.parse import urlparse
import emoji
import seaborn as sns
import random
import copy

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, recall_score,precision_score,roc_curve, auc
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertModel,BertTokenizer


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a folder for trained models
save_path = '/content/drive/MyDrive/DL_project/bert_violation_models/'
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Data loading and preprocess

## Data Formatting: The Triple-Input Approach
We structure the input as Rule [SEP] Subreddit [SEP] Comment. This allows the model to treat rule violation as a conditional task, evaluating the text specifically against the provided community guidelines.

In [ ]:
def add_examples_to_train_df(df):
    pos = df[["positive_example_1", "rule", "subreddit"]].rename(
        columns={"positive_example_1": "body"}
    )
    pos["rule_violation"] = 1

    pos_2 = df[["positive_example_2", "rule", "subreddit"]].rename(
        columns={"positive_example_2": "body"}
    )
    pos_2["rule_violation"] = 1

    neg = df[["negative_example_1", "rule", "subreddit"]].rename(
        columns={"negative_example_1": "body"}
    )
    neg["rule_violation"] = 0

    neg_2 = df[["negative_example_2", "rule", "subreddit"]].rename(
        columns={"negative_example_2": "body"}
    )
    neg_2["rule_violation"] = 0

    # combine
    df_add = pd.concat([pos, pos_2, neg, neg_2], ignore_index=True)
    df_add = df_add.dropna(subset=["body"]).reset_index(drop=True)
    df_add["rule_violation"] = df_add["rule_violation"].astype(int)

    return df_add

def create_input_seq(df):
    new_df = pd.DataFrame()
    add_to_df=None
    new_df["input"] = (
        "Rule: " + df["rule"] + " [SEP] " +
        "Subreddit: " + df["subreddit"] + " [SEP] " +
        "Comment: " + df["body"]
    )
    new_df["rule_violation"] = df["rule_violation"]
    return new_df

def process_url(text):
    urls = re.findall(r"(?:http|https)://[^\s]+", text)

    for url in urls:
        seen_semantics = set()
        all_semantics = []
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
        except ValueError:
            domain = "invalid"
        # domain
        domain_match = re.search(r"(?:https?://)?([a-z0-9\-\.]+)\.[a-z]{2,}", url.lower())
        if domain_match:
            full_domain = domain_match.group(1)
            parts = full_domain.split('.')
            for part in parts:
                if part and part not in seen_semantics and len(part) > 3:
                    all_semantics.append(f"domain:{part}")
                    seen_semantics.add(part)
         # path
        path = re.sub(r"^(?:https?://)?[a-z0-9\.-]+\.[a-z]{2,}/?", "", url.lower())
        path_parts = [p for p in re.split(r'[/_.-]+', path) if p and p.isalnum()]
        for part in path_parts:
            part_clean = re.sub(r"\.(html?|php|asp|jsp)$|#.*|\?.*", "", part)
            if part_clean and part_clean not in seen_semantics and len(part_clean) > 3:
                all_semantics.append(f"path:{part_clean}")
                seen_semantics.add(part_clean)

        if all_semantics:
            semantic_str = f"\n(URL Keywords: {' '.join(all_semantics)})"
        else:
            semantic_str = ""

        text = text.replace(url, semantic_str)

    return text


def clean_text(text):
    # create more clean text as input for the model
    text = process_url(text)
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def data_processing(df,add_examples=True):
    if add_examples:
        add_to_df = add_examples_to_train_df(df)
        df = pd.concat([df, add_to_df], ignore_index=True)

    df = create_input_seq(df)
    df['input'] = df['input'].apply(clean_text)
    return df

In [ ]:
# data path for local csv. file
data_path = '/content/drive/MyDrive/DL_project/train.csv'

In [ ]:
# Load the dataset
df = pd.read_csv(data_path)

# Clean and process input text and
df = data_processing(df)
X = df['input']
target = df['rule_violation']

## Data Exploration

In [ ]:
print("one example from train dataset:")
# Set column width to None (unlimited)
pd.set_option('display.max_colwidth', None)

# Optional: If you want to see more rows or columns too
pd.set_option('display.max_rows', 100)
print(df.head(1))

In [ ]:
# Plot the target distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='rule_violation', data=df)
plt.title('Class Distribution of Target Variable')
plt.show()

# Print the exact counts
print(df['rule_violation'].value_counts())


In [ ]:
# Function to split the preprocessed string
def extract_parts(row):
    parts = row.split('[SEP]')
    rule = parts[0].strip() if len(parts) > 0 else "Unknown"
    subreddit = parts[1].strip() if len(parts) > 1 else "Unknown"
    comment = parts[2].strip() if len(parts) > 2 else ""
    return pd.Series([rule, subreddit, comment])

# Create new columns for exploration
df_explore = df.copy()
df_explore[['rule', 'subreddit', 'comment']] = df['input'].apply(extract_parts)

In [ ]:
# Distribution of Rules in Dataset
plt.figure(figsize=(10, 6))
sns.countplot(data=df_explore, y='rule', order=df_explore['rule'].value_counts().index, palette='viridis')
plt.title('Distribution of Rules in Dataset')
plt.xlabel('Number of Samples')
plt.ylabel('Rule Name')
plt.show()

In [ ]:
# Create a cross-tabulation of Rule vs Label
plt.figure(figsize=(10, 8))
ct = pd.crosstab(df_explore['rule'], df_explore['rule_violation'])
ct_perc = ct.div(ct.sum(1), axis=0) # Normalize to percentages

ct_perc.plot(kind='barh', stacked=True, color=['#66b3ff','#ff9999'], figsize=(10, 6))
plt.title('Violation Rate (Label 1) per Rule')
plt.legend(title='Label', labels=['Clean (0)', 'Violation (1)'], loc='upper right')
plt.xlabel('Percentage of Samples')
plt.show()

In [ ]:
# Distribution of Comment Lengths
df_explore['comment_word_count'] = df_explore['comment'].apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 5))
sns.histplot(df_explore['comment_word_count'], bins=30, kde=True, color='purple')
plt.axvline(x=128, color='red', linestyle='--', label='BERT Limit (approx)')
plt.title('Distribution of Comment Lengths')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
# Perform  Split (10% for test or validation)
train_df, test_df, y_train, y_test = train_test_split(
    df,
    target,
    test_size=0.10,
    random_state=42
)

print(f"Training samples: {len(train_df)}")
print(f"Testing samples: {len(test_df)}")

# Set Tokenization for BERT

In [ ]:
# Initialize the BERT tokenizer
model_name = "google/bert_uncased_L-2_H-128_A-2"
tokenizer = BertTokenizer.from_pretrained(model_name)

def encode_comments(text_list, labels_list, max_len=128):
    encoded = tokenizer(
        text_list.tolist(),
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    return encoded['input_ids'], encoded['attention_mask'], torch.tensor(labels_list.values)

# Encode Training Data
train_ids, train_mask, train_y = encode_comments(train_df['input'], train_df['rule_violation'])

# Encode Testing Data
test_ids, test_mask, test_y = encode_comments(test_df['input'], test_df['rule_violation'])

# Create PyTorch DataLoaders
batch_size = 32

train_dataset = TensorDataset(train_ids, train_mask, train_y)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(test_ids, test_mask, test_y)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print("DataLoaders ready.")

In [ ]:
# Define device (move to GPU if possible)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Head Architectures:

## Architecture 1 – The Standard BERT Classifier
This model uses the standard [CLS] token output to generate a global summary of the text. It serves as our performance baseline for simple classification.

In [ ]:
class StandardBERTClassifier(nn.Module):
    def __init__(self, model_name=model_name, num_classes=1):
        super(StandardBERTClassifier, self).__init__()
        # Load the base BERT model
        self.bert = BertModel.from_pretrained(model_name)

        self.dropout = nn.Dropout(0.1)

        # The classification head: a single linear layer
        # Output is 1 for binary classification (using Sigmoid)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, input_ids, attention_mask):
        # 1. Feed input into BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # 2. Extract the [CLS] token representation
        # last_hidden_state shape: [batch_size, sequence_length, 128]
        # [CLS] is at index 0
        cls_output = outputs.last_hidden_state[:, 0, :]

        # 3. Regularization
        cls_output = self.dropout(cls_output)

        # 4. Pass to the linear head
        logits = self.classifier(cls_output)

        return logits

In [ ]:
# Initialize the model
model_standard = StandardBERTClassifier().to(device)
print("Standard BERT Model initialized.")

## Architecture 2 – BERT + CNN Head
To improve results, we add a 1D-Convolutional layer with multiple filter sizes (4, 6, 8, 10). This allows the model to "scan" for specific toxic phrases or n-gram patterns that might be lost in a global summary.

In [ ]:
class BERT_CNN_Classifier(nn.Module):
    def __init__(self, model_name=model_name, num_classes=1,n_filters = 100,filter_sizes = [4, 6, 8, 10]):
        super(BERT_CNN_Classifier, self).__init__()
        self.freeze_flag= True
        # Load the base BERT model
        self.bert = BertModel.from_pretrained(model_name)

        # Freeze bert param. while training
        for param in self.bert.parameters():
            param.requires_grad = False

        # CNN Parameters
        # n_filters: number of patterns to look for
        # filter_sizes: s (n-grams)
        # 128 * fs
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=128,
                      out_channels=n_filters,
                      kernel_size=fs)
            for fs in filter_sizes
        ])

        # 3. HYBRID POOLING (Max + Average = 2 * n_filters)
        self.fc = nn.Linear(len(filter_sizes) * n_filters * 2, num_classes)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        # 1. Get BERT embeddings
        if self.freeze_flag:
          with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            # last_hidden_state: [batch_size, seq_len, 128]
            embedded = outputs.last_hidden_state
        else:
          outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
          # last_hidden_state: [batch_size, seq_len, 128]
          embedded = outputs.last_hidden_state

        # 2. Reshape for CNN: [batch_size, 128, seq_len]
        embedded = embedded.permute(0, 2, 1)

        # 3. Apply Convolutions & Pooling
        # We apply Max Pooling to find the strongest "signal" in each filter
        conved = [F.relu(conv(embedded)) for conv in self.convs]
        # Hybrid Pooling: Max AND Average
        max_pooled = [F.max_pool1d(c, c.shape[2]).squeeze(2) for c in conved]
        avg_pooled = [F.avg_pool1d(c, c.shape[2]).squeeze(2) for c in conved]

        # 4. Concatenate the max and ave. pooled features from different filter sizes
        cat = self.dropout(torch.cat(max_pooled+avg_pooled, dim=1))

        # 5. Final Classification
        return self.fc(cat)

    def unfreeze_bert(self):
          print("Unfreezing BERT layers for fine-tuning...")
          self.freeze_flag = False
          for param in self.bert.parameters():
              param.requires_grad = True

In [ ]:
# Initialize the model
model_cnn = BERT_CNN_Classifier().to(device)
print("BERT-CNN Model initialized.")

## Architecture 3 – BERT + Attention Head
This head uses a Trainable Attention mechanism to assign a "weight of importance" to every word. This is our primary tool for Explainability, as it tells us exactly which words (e.g., "spam", "links") led to a "Violation" verdict.

In [ ]:
class BERT_Attention_Model(nn.Module):
    def __init__(self, model_name=model_name, num_classes=1):
        super(BERT_Attention_Model, self).__init__()
        self.freeze_flag= True
        # Load the base BERT model
        self.bert = BertModel.from_pretrained(model_name)

        # Freeze bert param. while training
        for param in self.bert.parameters():
            param.requires_grad = False

        # layer "scores" each word's importance
        self.attention_net = nn.Sequential(
            nn.Linear(128, 64), # Project hidden state to a smaller space
            nn.Tanh(),           # Non-linearity
            nn.Linear(64, 1)    # Output a single score per word
        )

        # final classifier
        self.fc = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        # 1. Get BERT embeddings
        if self.freeze_flag:
          with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            # last_hidden_state: [batch_size, seq_len, 128]
            embedded = outputs.last_hidden_state
        else:
          outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
          # last_hidden_state: [batch_size, seq_len, 128]
          embedded = outputs.last_hidden_state

        # Step 1: Calculate attention weights for every word
        # weights shape: [Batch, Seq_Len, 1]
        weights = self.attention_net(embedded)

        # Step 2: Mask out the padding tokens so they don't get attention
        # We fill padding positions with a very large negative number
        weights = weights.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e9)

        # Step 3: Softmax makes all weights sum to 1.0 (100%)
        # soft_weights shape: [Batch, Seq_Len, 1]
        soft_weights = F.softmax(weights, dim=1)
        self.attn_weights = soft_weights

        # Step 4: Weighted Sum - multiply each word by its weight and sum them up
        # result shape: [Batch, 128]
        attended_representation = torch.sum(soft_weights * embedded, dim=1)

        # Step 5: Classify the final "Attended" vector
        logits = self.fc(self.dropout(attended_representation))

        return logits

    def unfreeze_bert(self):
          print("Unfreezing BERT layers for fine-tuning...")
          self.freeze_flag = False
          for param in self.bert.parameters():
              param.requires_grad = True

In [ ]:
# Initialize the model
model_atten = BERT_Attention_Model().to(device)
print("BERT-Attention Model initialized.")

# Training

In [ ]:
# Define utility function for metrics calculation
def calculate_metrics(all_preds, all_labels):
    # Convert logits to probabilities (0-1) and then to binary (0 or 1)
    probs = torch.sigmoid(torch.tensor(all_preds)).numpy()
    preds = (probs >= 0.5).astype(int)
    labels = np.array(all_labels)

    return {
        "acc": accuracy_score(labels, preds),
        "recall": recall_score(labels, preds),
        "precision": precision_score(labels, preds),
        "f1": f1_score(labels, preds),
        "auc": roc_auc_score(labels, probs)
    }

In [ ]:

def train_model(model, train_loader, test_loader, optimizer, criterion,
                epochs=5, unfreeze_at=0, lr_change=2e-5,
                patience=2, min_delta=0.01,is_early_stop=False):

    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_auc": []}

    # Early Stopping Variables
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    early_stop_triggered = False

    for epoch in range(epochs):
        # 1. Handle Unfreezing Logic
        if unfreeze_at and epoch == unfreeze_at:
            model.unfreeze_bert()
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr_change)
            print(f"\n--- BERT Unfrozen: Learning rate set to {lr_change} ---")
            # Reset early stopping best_loss when model architecture/LR changes significantly
            best_val_loss = float('inf')
            patience_counter = 0

        # 2. Training Phase
        model.train()
        total_train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

        for batch in pbar:
            ids, mask, labels = [t.to(device) for t in batch]
            optimizer.zero_grad()

            logits = model(ids, mask).squeeze()
            loss = criterion(logits, labels.float())

            loss.backward()
            # Recommended: Add gradient clipping to prevent BERT from "exploding"
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            total_train_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_train_loss = total_train_loss / len(train_loader)
        history["train_loss"].append(avg_train_loss)

        # 3. Evaluation Phase
        model.eval()
        all_preds, all_labels = [], []
        total_val_loss = 0

        with torch.no_grad():
            for batch in test_loader:
                ids, mask, labels = [t.to(device) for t in batch]
                logits = model(ids, mask).squeeze()

                loss = criterion(logits, labels.float())
                total_val_loss += loss.item()

                all_preds.extend(logits.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())

        avg_val_loss = total_val_loss / len(test_loader)
        metrics = calculate_metrics(all_preds, all_labels)

        history["val_loss"].append(avg_val_loss)
        history["val_f1"].append(metrics['f1'])
        history["val_auc"].append(metrics['auc'])

        print(f"\nEpoch {epoch+1} Results: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"F1: {metrics['f1']:.4f} | AUC: {metrics['auc']:.4f}")

        # 4. Early Stopping & Checkpointing Logic
        # We check if the improvement is better than our min_delta
        if is_early_stop:
          if avg_val_loss < (best_val_loss - min_delta):
              best_val_loss = avg_val_loss
              best_model_wts = copy.deepcopy(model.state_dict()) # Save the best version
              patience_counter = 0
              print(f" Validation loss improved. Saving best model.")
          else:
              patience_counter += 1
              print(f" No improvement for {patience_counter} epoch(s).")

          if patience_counter >= patience:
              print(f"\n Early stopping triggered at epoch {epoch+1}!")
              early_stop_triggered = True
              break
        else:
          best_model_wts = copy.deepcopy(model.state_dict()) # Save the best version

    # Load the best weights back into the model before returning
    model.load_state_dict(best_model_wts)
    print("Final Model: Best weights restored.")

    return history, model

## Comparbility
We tried to keep the training process by parameters is much equal as possible between different models in order to maintain comparbility in evaluation.
still the learning rate was changed in order to fit the random init.

In [ ]:
# Setting same optimizer type
optimizer_standard = torch.optim.AdamW(model_standard.parameters(), lr=2e-5)
# In CNN head & Attention head -
# must be initialized with higher learning rate when trying to train randomly init. CNN & Attention weights
optimizer_cnn = torch.optim.AdamW(model_cnn.parameters(), lr=1e-3)
optimizer_atten = torch.optim.AdamW(model_atten.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()
EPOCHS = 6
Continue_Training= False

In [ ]:
# Load models if trained already for procceed training
if Continue_Training:
  if os.path.exists(save_path + 'bert_cnn_tiny.pth'):
    # Load the weights from the file
    try:
      model_cnn.load_state_dict(torch.load(save_path + 'bert_cnn_tiny.pth'))
      print("Model CNN loaded for continue training")
    except:pass
  if os.path.exists(save_path + 'standard_bert_tiny.pth'):
    # Load the weights from the file
    try:
      model_standard.load_state_dict(torch.load(save_path + 'standard_bert_tiny.pth'))
      print("Model Standart loaded for continue training")
    except:pass
  if os.path.exists(save_path + 'bert_atten_tiny.pth'):
    # Load the weights from the file
    try:
      model_atten.load_state_dict(torch.load(save_path + 'bert_atten_tiny.pth'))
      print("Model Atten loaded for continue training")
    except:pass

In [ ]:
# 1. Train Standard BERT (Full Fine-Tuning)
print("--- Training Standard BERT-Tiny (Linear Head) ---")
# Standard BERT is unfrozen by default in our class
standard_history,model_standard = train_model(model_standard, train_loader, test_loader, optimizer_standard, criterion,epochs=EPOCHS,is_early_stop=False)
# Save the Standard BERT-Tiny Model
torch.save(model_standard.state_dict(), save_path + 'standard_bert_tiny.pth')
print(f"Standard BERT-Tiny Model saved successfully")

## Training Strategy
To prevent "scrambling" the pre-trained BERT weights with the randomly initialized CNN/Attention heads, we tried using two stages:
Warmup: Freeze BERT and train only the head
Fine-Tuning: Unfreeze BERT and train the entire system with a much lower learning rate.
**Evantually The two stages approche was worst and so we kept BERT frozen for the whole training**

In [ ]:
# 2. Train BERT-CNN (Frozen)
print("\n--- Training BERT-Tiny + CNN Head ---")
cnn_history,model_cnn = train_model(model_cnn, train_loader, test_loader, optimizer_cnn, criterion,epochs=EPOCHS,is_early_stop=False)
# Save the BERT-CNN Model
torch.save(model_cnn.state_dict(), save_path + 'bert_cnn_tiny.pth')
print(f"BERT-Tiny + CNN Head Model saved successfully")

In [ ]:
# 2. Train BERT-CNN (Two stages strategy ) - Disabled
# print("\n--- Training BERT-Tiny + CNN Head ---")
# model_cnn_2 = BERT_CNN_Classifier().to(device)
# optimizer_cnn_2 = torch.optim.AdamW(model_cnn.parameters(), lr=1e-3)
# _,model_cnn_2 = train_model(model_cnn_2, train_loader, test_loader, optimizer_cnn_2, criterion,epochs=20,unfreeze_at=3)

In [ ]:
# 3. Train BERT-Attention (Frozen)
print("\n--- Training BERT-Tiny + Attention Head ---")
atten_history,model_atten = train_model(model_atten, train_loader, test_loader, optimizer_atten, criterion,epochs=EPOCHS,is_early_stop=False)
# Save the BERT-Attention Model
torch.save(model_atten.state_dict(), save_path + 'bert_atten_tiny.pth')
print(f"BERT-Tiny + Attention Head Model saved successfully")

In [ ]:
# 3. Train BERT-Attention (Two stages strategy ) - Disabled
# print("\n--- Training BERT-Tiny + Attention Head ---")
# model_atten_2 = BERT_Attention_Model().to(device)
# optimizer_atten_2 = torch.optim.AdamW(model_atten.parameters(), lr=1e-3)
# _,model_atten_2 = train_model(model_atten_2, train_loader, test_loader, optimizer_atten_2, criterion,epochs=20,unfreeze_at=6)

# Evaluation

### Load models for evaluation :

In [ ]:
loaded_model_cnn = BERT_CNN_Classifier().to(device)
# Load the weights from the file
loaded_model_cnn.load_state_dict(torch.load(save_path + 'bert_cnn_tiny.pth'))
# Set to evaluation mode
loaded_model_cnn.eval()
pass

In [ ]:
loaded_model_stan = StandardBERTClassifier().to(device)
# Load the weights from the file
loaded_model_stan.load_state_dict(torch.load(save_path + 'standard_bert_tiny.pth'))
# Set to evaluation mode
loaded_model_stan.eval()
pass

In [ ]:
loaded_model_atten = BERT_Attention_Model().to(device)
# Load the weights from the file
loaded_model_atten.load_state_dict(torch.load(save_path + 'bert_atten_tiny.pth'))
# Set to evaluation mode
loaded_model_atten.eval()
pass

In [ ]:
def evaluate_on_test(model, test_loader, model_name="Model"):
    model.eval()
    all_preds = []
    all_labels = []

    print(f"Running final evaluation for: {model_name}...")

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            ids, mask, labels = [t.to(device) for t in batch]
            logits = model(ids, mask).squeeze()

            # If batch size is 1, squeeze can remove too many dims; handle gracefully
            if logits.dim() == 0:
                logits = logits.unsqueeze(0)

            all_preds.extend(logits.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return all_preds,all_labels


In [ ]:
# Execute evaluation for all models
all_preds_stand,all_labels = evaluate_on_test(loaded_model_stan, test_loader, "Standard BERT-Tiny")
all_preds_cnn,_  = evaluate_on_test(loaded_model_cnn, test_loader, "BERT-Tiny + CNN Head")
all_preds_atten,_  = evaluate_on_test(loaded_model_atten, test_loader, "BERT-Tiny + Attention Head")

In [ ]:
# Use the calculate_metrics function
standard_results = calculate_metrics(all_preds_stand, all_labels)
cnn_results = calculate_metrics(all_preds_cnn, all_labels)
atten_results = calculate_metrics(all_preds_atten, all_labels)
# Create a Comparison Table
comparison_df = pd.DataFrame([standard_results, cnn_results,atten_results],
                             index=["Standard BERT (Linear)", "BERT + CNN (Begin. Frozen)","BERT + Attention (Begin. Frozen)"])

print("\n" + "="*30)
print("FINAL TEST DATASET RESULTS")
print("="*30)
print(comparison_df.round(4))

In [ ]:
def plot_test_roc(models_dict, test_loader):
    plt.figure(figsize=(8, 6))

    for name, model in models_dict.items():
        model.eval()
        probs = []
        labels = []
        with torch.no_grad():
            for batch in test_loader:
                ids, mask, lbls = [t.to(device) for t in batch]
                logits = model(ids, mask).squeeze()
                probs.extend(torch.sigmoid(logits).cpu().tolist())
                labels.extend(lbls.cpu().tolist())

        fpr, tpr, _ = roc_curve(labels, probs)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve Comparison (Test Set)')
    plt.legend()
    plt.show()



In [ ]:
# Plot Roc curve
plot_test_roc({
    "Standard BERT": model_standard,
    "BERT + CNN": model_cnn,
    "BERT + Attention": model_atten
}, test_loader)

In [ ]:
plt.figure(figsize=(12, 5))

# Plot F1-Score comparison
plt.subplot(1, 2, 1)
plt.plot(standard_history['val_f1'], label='Standard BERT (Linear)')
plt.plot(cnn_history['val_f1'], label='BERT + CNN (Frozen)')
plt.plot(atten_history['val_f1'], label='BERT + Atten (Frozen)')
plt.title('Validation F1-Score')
plt.xlabel('Epoch')
plt.legend()

# Plot Loss comparison
plt.subplot(1, 2, 2)
plt.plot(standard_history['train_loss'], label='Standard BERT (Linear)')
plt.plot(cnn_history['train_loss'], label='BERT + CNN (Frozen)')
plt.plot(atten_history['train_loss'], label='BERT + Atten (Frozen)')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.legend()

plt.show()

In [ ]:
def show_model_examples(results, original_texts, num_examples=1):

    # 2. Categorize results
    correct = [r for r in results if r['true'] == r['pred']]
    fp = [r for r in results if r['true'] == 0 and r['pred'] == 1] # False Positives
    fn = [r for r in results if r['true'] == 1 and r['pred'] == 0] # False Negatives

    # 3. Print Function
    def print_list(title, sample_list):
        print(f"\n{'='*20} {title} {'='*20}")
        for i, item in enumerate(sample_list[:num_examples]):
          print(f"\nExample {i+1}:")
          print(f"Text: {item['text']}...") # Showing first 300 chars
          print(f"True Label: {item['true']} | Predicted: {item['pred']}")

    # Display results
    print_list("MISTAKES: FALSE POSITIVES (Flagged clean text)", fp)
    print_list("MISTAKES: FALSE NEGATIVES (Missed a violation)", fn)



In [ ]:
# Show examples of correct and imcorrect

probs = torch.sigmoid(torch.tensor(all_preds_stand)).numpy()
preds_stan = (probs >= 0.5).astype(int)

probs = torch.sigmoid(torch.tensor(all_preds_cnn)).numpy()
preds_cnn = (probs >= 0.5).astype(int)

test_texts = test_df['input'].tolist()
examples_dict_cnn=[]
for i in range(len(all_preds_cnn)):
    examples_dict_cnn.append({
        'text': test_texts[i],
        'true': all_labels[i],
        'pred': preds_cnn[i],
    })

examples_dict_stan=[]
for i in range(len(all_preds_stand)):
    examples_dict_stan.append({
        'text': test_texts[i],
        'true': all_labels[i],
        'pred': preds_stan[i],
    })


print("Examples for Standard BERT:")
show_model_examples(examples_dict_stan, test_texts)

print("\n\nExamples for BERT-CNN:")
show_model_examples(examples_dict_cnn, test_texts)

### CNN Head visulzation:

The following function accesses the raw weight tensors of my 1D-CNN kernels. It visualizes the internal "pattern-matching" parameters the model has learned.
**Future Work** might be applied in Model Pruning By identifying filters with near-zero or redundant weights, you can remove them to create a "lighter" version of the model for mobile or edge devices.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_filter_weights(model, filter_size_idx=0, filter_idx=0):
    # Access the specific conv layer (e.g., filter_size_idx 0 is size 2)
    weights = model.convs[filter_size_idx].weight.data.cpu().numpy()

    # Shape of weights: [n_filters, 128, kernel_size]
    # We take one specific filter
    single_filter = weights[filter_idx]

    plt.figure(figsize=(8, 10))
    sns.heatmap(single_filter, cmap="RdBu", center=0)
    plt.title(f"Weights for Filter {filter_idx} (Size {model.convs[filter_size_idx].kernel_size[0]})")
    plt.xlabel("Word Position in N-gram")
    plt.ylabel("BERT Embedding Dimensions (128)")
    plt.show()

# Example: Plot the first filter of the size-3 conv layer
plot_filter_weights(model_cnn, filter_size_idx=0, filter_idx=1)

This tool identifies the specific spans of text (n-grams) that produced the highest numerical activation for a given filter.
**Future Work** Can be applied by Automated Rule Refining: Subreddit moderators can use this to see if the model is consistently flagging specific phrases that shouldn't be violations, allowing them to refine the community rules.

In [ ]:
def get_top_activating_ngrams(model, tokenizer, test_loader, filter_size_idx=1, filter_idx=0, top_k=5):
    model.eval()
    activations = []

    kernel_size = model.convs[filter_size_idx].kernel_size[0]

    with torch.no_grad():
        for batch in test_loader:
            ids, mask, _ = [t.to(device) for t in batch]

            # Get BERT embeddings
            outputs = model.bert(input_ids=ids, attention_mask=mask)
            embedded = outputs.last_hidden_state.permute(0, 2, 1) # [B, 128, Seq]

            # Get conv output for the specific filter size
            # conv_out shape: [Batch, 100, Seq_len - kernel + 1]
            conv_out = F.relu(model.convs[filter_size_idx](embedded))

            # Focus on the specific filter_idx
            # specific_filter shape: [Batch, Seq_len - kernel + 1]
            specific_filter = conv_out[:, filter_idx, :]

            for b in range(ids.shape[0]):
                # Find the position that had the max activation for this sentence
                max_val, max_pos = torch.max(specific_filter[b], dim=0)

                # Retrieve the original tokens for that n-gram
                ngram_ids = ids[b, max_pos:max_pos + kernel_size]
                ngram_text = tokenizer.decode(ngram_ids)
                # Reconstruct the full text for readability
                full_text = tokenizer.decode(ids[b], skip_special_tokens=True)

                activations.append((max_val.item(), ngram_text,full_text))

    # Sort by activation strength
    activations.sort(key=lambda x: x[0], reverse=True)

    print(f"Top {top_k} phrases for Filter {filter_idx} (Size {kernel_size}):")
    for val, text,fulltext in activations[:top_k]:
        print(f"Full Text: {fulltext[:120]}...")
        print(f"Score: {val:.4f} | Text: '{text}'")


In [ ]:
# Example: See what filter n is looking for
get_top_activating_ngrams(loaded_model_cnn, tokenizer, test_loader, filter_size_idx=2, filter_idx=10)

### Attention head visulzation:

This function extracts the "soft weights" from the Attention head and maps them back onto the original token so it plot the "importance" the model assigned to five highest value words when making its final verdict.
**Future Work** - same Automated Rule Refining like mentioned above

In [ ]:
def visualize_attention(model, tokenizer, test_loader, device, num_samples=1):
    model.eval()

    # 1. Grab one batch from the test loader
    # We use iter() and next() to get just the first batch
    batch = next(iter(test_loader))

    # Unpack the batch (assuming format: ids, mask, labels)
    input_ids = batch[0].to(device)
    attention_mask = batch[1].to(device)
    labels = batch[2].to(device)

    # 2. Forward pass to get weights
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        preds = (probs > 0.5).astype(int)
        # Weights shape: [Batch, Seq_Len, 1]
        weights = model.attn_weights.squeeze(-1).cpu().numpy()

    # Move to CPU for processing
    input_ids = input_ids.cpu().numpy()


    # 3. Process the requested number of samples
    for i in range(min(num_samples, input_ids.shape[0])):
        k = random.randint(0,min(num_samples, input_ids.shape[0]))
        sample_ids = input_ids[k]
        sample_weights = weights[k]

        # Metadata for this specific sample
        prob_score = probs[k]
        pred_label = preds[k]
        actual_label = labels[k]

        if actual_label ==0 and pred_label==0: continue

        # Decode tokens to handle word-piece splits correctly
        tokens = tokenizer.convert_ids_to_tokens(sample_ids)

        # Reconstruct the full text for readability
        full_text = tokenizer.decode(sample_ids, skip_special_tokens=True)

        print(f"\n--- Sample {i+1} ---")
        print(f"Full Text: {full_text[:120]}...")
        print(f"Actual Label: {'Violation' if labels[i].item() == 1 else 'No Violation'}")
        # Print Summary
        print(f"\n--- Sample {i+1} ---")
        print(f"Prediction: {'VIOLATION' if pred_label == 1 else 'No Violation'}")
        print(f"Actual:     {'VIOLATION' if actual_label == 1 else 'No Violation'}")
        # Create token-weight pairs (ignoring padding)
        valid_pairs = []
        for token, weight in zip(tokens, sample_weights):
            if token == '[PAD]': continue
            valid_pairs.append((token, weight))

        # Sort to find the "Top Evidence"
        valid_pairs.sort(key=lambda x: x[1], reverse=True)

        print(f"{'Token':<15} | {'Attention Score':<15}")
        print("-" * 35)
        for token, weight in valid_pairs[:5]: # Show top 8 influencers
            print(f"{token:<15} | {weight:.4f}")


In [ ]:
# Example: See on few samples which words the attention head gave the highest weights
visualize_attention(loaded_model_atten, tokenizer, test_loader, device, num_samples=5)